In [1]:
import Data.List (sortOn)

type Estado = (Int, Int)
type Nodo   = Estado
type Costo  = Int
type Grafo  = [(Nodo, [(Nodo, Costo)])]

capA, capB :: Int
capA = 5
capB = 3

In [2]:
transferirAB :: Estado -> Estado
transferirAB (a, b) =
  let transferido = min a (capB - b)
  in  (a - transferido, b + transferido)

transferirBA :: Estado -> Estado
transferirBA (a, b) =
  let transferido = min b (capA - a)
  in  (a + transferido, b - transferido)

In [9]:
vecinos :: Estado -> [(Estado, Costo)]
vecinos (a, b) =
  [ ((capA, b),          1)   
  , ((a, capB),          1)   
  , ((0, b),             1)  
  , ((a, 0),             1)  
  , (transferirAB (a,b), 1)   
  , (transferirBA (a,b), 1)   
  ]

In [4]:
heuristica :: Estado -> Costo
heuristica (a, b)
  | a == 4 || b == 4 = 0
  | otherwise        = 1

In [5]:
aStar :: (Nodo -> [(Nodo, Costo)]) -> (Nodo -> Costo) -> Nodo -> Nodo -> [(Nodo, Costo)]
aStar sucesores heuristica inicio meta = buscar [(inicio, 0)] []
  where
    buscar [] _          = []
    buscar ((nodo, costo):cola) visitados
      | nodo == meta     = [(nodo, costo)]
      | nodo `elem` visitados = buscar cola visitados
      | otherwise        =
          let nuevos    = [ (v, costo + c) | (v, c) <- sucesores nodo
                                           , v `notElem` visitados ]
              ordenados = sortOn (\(v, c) -> c + heuristica v) (cola ++ nuevos)
          in  (nodo, costo) : buscar ordenados (nodo : visitados)

In [10]:
inicio = (0, 0)
meta   = (4, 0)  

resultado = aStar vecinos heuristica inicio meta

main :: IO ()
main = do
  putStrLn "Camino encontrado (estado, costo acumulado):"
  mapM_ print resultado
  putStrLn $ "\nTotal de pasos: " ++ show (length resultado - 1)

In [11]:
let resultado = aStar vecinos heuristica (0,0) (4,3)
mapM_ print resultado

((0,0),0)
((5,0),1)
((0,3),1)
((5,3),2)
((2,3),2)
((3,0),2)
((2,0),3)
((3,3),3)
((0,2),4)
((5,1),4)
((5,2),5)
((0,1),5)
((4,3),6)